In [ ]:
#!/usr/bin/env python3
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------
# Planet parameters
# -----------------------------
RM = 2440
R_CORE = 2080

# -----------------------------
# Sphere mesh
# -----------------------------
theta = np.linspace(0, np.pi, 80)
phi = np.linspace(0, 2*np.pi, 120)
theta, phi = np.meshgrid(theta, phi)

def sphere(R):
    x = R * np.sin(theta) * np.cos(phi)
    y = R * np.sin(theta) * np.sin(phi)
    z = R * np.cos(theta)
    return x, y, z

# -----------------------------
# Cutaway function (upper wedge)
# -----------------------------
def cutaway(x, y, z):
    phi = np.arctan2(y, x)
    mask = (z > 0) & (phi > 0) & (phi < np.pi/4)
    return np.where(mask, np.nan, x), np.where(mask, np.nan, y), np.where(mask, np.nan, z)

# -----------------------------
# Lit surface
# -----------------------------
def lit_surface(x, y, z, color):
    return go.Surface(
        x=x, y=y, z=z,
        surfacecolor=np.zeros_like(x),
        colorscale=[[0,color],[1,color]],
        showscale=False,
        lighting=dict(
            ambient=0.6,
            diffuse=0.8,
            specular=0.2,
            roughness=0.9,
            fresnel=0.1
        ),
        lightposition=dict(x=3, y=0, z=0)  # fixed light along +X in RM units
    )

# -----------------------------
# Sphere coordinates (in RM units!)
# -----------------------------
x_solid, y_solid, z_solid = [coord / RM for coord in sphere(RM)]
x_outer, y_outer, z_outer = [coord / RM for coord in sphere(RM)]
x_inner, y_inner, z_inner = [coord / RM for coord in sphere(R_CORE)]

# Apply cutaway to outer shell only
x_outer_cut, y_outer_cut, z_outer_cut = cutaway(x_outer, y_outer, z_outer)

# Inner core (no cutaway)
x_core, y_core, z_core = x_inner, y_inner, z_inner

# -----------------------------
# Create figure
# -----------------------------
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type":"scene"}, {"type":"scene"}]],
    subplot_titles=["Resistive Body", "Conductive Core"]
)

# -----------------------------
# Panel 1: Solid planet
# -----------------------------
fig.add_trace(lit_surface(x_solid, y_solid, z_solid, "coral"), row=1, col=1)

# -----------------------------
# Panel 2: Cutaway shell + core
# -----------------------------
fig.add_trace(lit_surface(x_outer_cut, y_outer_cut, z_outer_cut, "coral"), row=1, col=2)
fig.add_trace(lit_surface(x_core, y_core, z_core, "lightblue"), row=1, col=2)

# -----------------------------
# Set identical axes and aspect ratio
# -----------------------------
for i in range(1,3):
    scene = "scene" if i==1 else f"scene{i}"
    fig.layout[scene].update(
        xaxis=dict(range=[-1.5, 1.5], title="X [Rₘ]", showgrid=True),
        yaxis=dict(range=[-1.5, 1.5], title="Y [Rₘ]", showgrid=True),
        zaxis=dict(range=[-1.5, 1.5], title="Z [Rₘ]", showgrid=True),
        aspectmode="cube",
        camera=dict(
            eye=dict(x=2, y=2, z=1.5),  # slightly outside
            center=dict(x=0, y=0, z=0)
        )
    )

fig.update_layout(width=1200, height=800, title="Mercury Interior Configurations")

fig.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

out_dir = "/Users/danywaller/Projects/mercury/extreme/"

# -----------------------------
# Parameters
# -----------------------------
R_PLANET = 2      # Mask radius in Rm units
EXTENT = 5        # Axis limits in Rm units
N_POINTS = 150    # Resolution of grid

# -----------------------------
# IMF vectors (normalized)
# -----------------------------
vA = np.array([-24, 0, 7]); vA = vA/np.linalg.norm(vA)
vB = np.array([-24, 0, -7]); vB = vB/np.linalg.norm(vB)

# -----------------------------
# Create X-Z grid
# -----------------------------
x = np.linspace(-EXTENT, EXTENT, N_POINTS)
z = np.linspace(-EXTENT, EXTENT, N_POINTS)
X, Z = np.meshgrid(x, z)

# -----------------------------
# Function to compute 2D projected vectors in X-Z plane
# -----------------------------
def vector_field_2d(vec, X, Z, mask_radius=R_PLANET):
    """
    Project 3D vector onto X-Z plane.
    Mask points inside a circle of radius mask_radius centered at (0,0).
    """
    # Repeat vec components in 2D
    U = np.full_like(X, vec[0])
    W = np.full_like(Z, vec[2])

    # Mask inside planet
    r = np.sqrt(X**2 + Z**2)
    U[r < mask_radius] = np.nan
    W[r < mask_radius] = np.nan

    return U, W

# -----------------------------
# Compute vector fields
# -----------------------------
U_A, W_A = vector_field_2d(vA, X, Z)
U_B, W_B = vector_field_2d(vB, X, Z)

# -----------------------------
# Plot
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(12,6))
planes = [(U_A, W_A, "Planetward-Southward"), (U_B, W_B, "Planetward-Northward")]

clrs = ['mediumorchid', 'limegreen']

for ax, (U, W, label), clr in zip(axes, planes, clrs):
    # Streamlines
    ax.streamplot(X, Z, U, W, color=clr, linewidth=2.0, density=1.0, arrowsize=2.2)
    # Planet mask
    planet = plt.Circle((0,0), R_PLANET, color='coral')
    ax.add_patch(planet)
    ax.set_aspect('equal')
    ax.set_xlim(-EXTENT, EXTENT)
    ax.set_ylim(-EXTENT, EXTENT)
    ax.set_xlabel('X [Rₘ]', fontsize=12)
    ax.set_ylabel('Z [Rₘ]', fontsize=12)
    ax.set_title(f' {label} IMF', fontsize=12)

plt.suptitle("Mercury IMF Field Lines (X-Z plane)", y=1.01, fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()
out_png = os.path.join(out_dir, f"mercury_imf_orientation_cases.png")
fig.savefig(out_png, dpi=250, bbox_inches='tight')